In [11]:
# Create markdown cells to document processes 

Windows_prefix="../Kaggle_Data/titanic/"
MacOS_prefix="../Kaggle_Data/"

In [12]:
# Variables defined

# survival	Survival	0 = No, 1 = Yes
# pclass	Ticket class	1 = 1st, 2 = 2nd, 3 = 3rd
# sex	Sex	
# Age	Age in years	
# sibsp	# of siblings / spouses aboard the Titanic	
# parch	# of parents / children aboard the Titanic	
# ticket	Ticket number	
# fare	Passenger fare	
# cabin	Cabin number	
# embarked	Port of Embarkation	C = Cherbourg, Q = Queenstown, S = Southampton

In [13]:
import pandas as pd

print(pd.__version__)

3.0.6


In [14]:
# Reading in the Data and looking at the first 10 rows
# Data can be found at 
# survival = pd.read_csv("../Kaggle_Data/gender_submission.csv")

# perhaps prefixes

# Windows path different will need to update how it finds the csv
survival = pd.read_csv(Windows_prefix+"gender_submission.csv")
survival.head(10)

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [15]:
#Check for NaNs 
survival[survival.isnull().any(axis=1)]
#No Nans in the survival table

,PassengerId,Survived


In [73]:
train_data = pd.read_csv(Windows_prefix+"train.csv")
train_data.head(10)
train_data = train_data.drop(columns="Name")

train_null_rows = train_data[train_data.isnull().any(axis=1)].shape[0]
# We find that 331 rows have null values
total_train_rows = train_data.shape[0]


train_null_rows / total_train_rows
# 79.5% of the rows have null values, so it might be important to check which specific columns have thes null values, may be wise to drop that column later on

0.7946127946127947

In [ ]:
#Merging the test dataframe to have the survival as well

# --------------------------------------------------------------------------------------------------------------------------------------------------------------------------#

In [ ]:
#Dropping names, do not think it would be a huge factor

# test_with_survival_df = test_with_survival_df.drop(columns="Name")

In [ ]:
train_data

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,male,26.0,0,0,111369,30.0000,C148,C


In [75]:
# One hot encoding on Sex, Pclass, and Embarked
full_df = pd.get_dummies(train_data, columns=["Sex","Pclass","Embarked"],dtype="int",drop_first=True)
full_df


,PassengerId,Survived,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Pclass_2,Pclass_3,Embarked_Q,Embarked_S
0,1,0,22.0,1,0,A/5 21171,7.2500,NaN,1,0,1,0,1
1,2,1,38.0,1,0,PC 17599,71.2833,C85,0,0,0,0,0
2,3,1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0,0,1,0,1
3,4,1,35.0,1,0,113803,53.1000,C123,0,0,0,0,1
4,5,0,35.0,0,0,373450,8.0500,NaN,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,27.0,0,0,211536,13.0000,NaN,1,1,0,0,1
887,888,1,19.0,0,0,112053,30.0000,B42,0,0,0,0,1
888,889,0,NaN,1,2,W./C. 6607,23.4500,NaN,0,0,1,0,1
889,890,1,26.0,0,0,111369,30.0000,C148,1,0,0,0,0


In [76]:
#finding out pct NA for Age and Cabin

In [77]:
# Doing a base level of analysis without adding any features, with diff models.
# First is sklearn

from sklearn.linear_model import LogisticRegression
#utilizing logistic regression since the y / dependent variable is survived or not survived

LogReg_y = full_df["Survived"]
#dropping Tickets, and Cabin columns since they have yet to be converted to dummies and they are not numerical. May consider them again
LogReg_X = full_df.drop(columns=["Survived","Ticket","Cabin"])

# LogReg_X
# clf = LogisticRegression(random_state=0).fit(LogReg_X, LogReg_y)


In [78]:
# Our age column has loads of NaNs in it, we cannot do the regression just yet since it doesn't really deal with Nans

In [79]:
#May have to look for nans in each row, also look into the unique cabins

#Looking through the full_df around other columns to check for repeatability

full_df["Cabin"].value_counts()
#The highest mode is 3

#Investigating where cabin numbers repeated most
full_df[full_df["Cabin"] == "B57 B59 B63 B66"]
#This most likely means they are in the same Cabin, maybe there is a way to check if number of people in a cabin was a factor (Maybe it confounds with sibsp or parch, can check later)



#Is there a way to map their ticket or some other variable onto their position on the ship

#Based of Cabin positions it seems Upper Deck was E, Saloon deck D, Shelter Deck C, Bridge deck B

,PassengerId,Survived,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Pclass_2,Pclass_3,Embarked_Q,Embarked_S
311,312,1,18.0,2,2,PC 17608,262.375,B57 B59 B63 B66,0,0,0,0,0
742,743,1,21.0,2,2,PC 17608,262.375,B57 B59 B63 B66,0,0,0,0,0
